# Northgate AI Stock Predictor â€” Exploratory Data Analysis

*PRD Section 6.* Each analysis ends in a written observation and the modelling decision it justified. Generated by `src/create_eda.py`; every figure is saved under `reports/figures/`.

In [ ]:
import sys
from pathlib import Path
# Resolved relative to the notebook, not hardcoded. An absolute path here
# writes the build machine's username and folder layout into a committed
# artifact and makes the notebook fail for anyone else who opens it.
ROOT = Path.cwd()
if not (ROOT / 'config.yaml').exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from src.common import CLEANED_PANEL, load_config
from src import math_utils as mu

cfg = load_config()
PRICE = 'Adjusted Close'
panel = pd.read_parquet(CLEANED_PANEL)
panel['Date'] = pd.to_datetime(panel['Date'])
UNIVERSE = ['AAPL', 'MSFT', 'JPM', 'XOM', 'JNJ', 'PG', 'NVDA', 'KO', 'CAT', 'HD']
BENCH, VIX = '^GSPC', '^VIX'
panel.head()

## Trend analysis

**Method** â€” Price with 50/200-day MAs; log-scale view

**Finding** â€” All 6 sampled series rise over the window; the normalised paths are clearly non-stationary, and price oscillates around its moving averages rather than fluctuating around a constant level.

**Modelling decision** â€” Model the forward RETURN, not the price level. Target fixed in config.yaml as a 1-session forward log return of Adjusted Close; this also keeps the scaler meaningful, since a raw price level would not be stationary across splits.


In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))
for t in [BENCH, 'AAPL', 'MSFT', 'JPM', 'XOM', 'NVDA']:
    s = panel[panel['Ticker'] == t].sort_values('Date')
    ax.plot(s['Date'], s[PRICE] / s[PRICE].iloc[0], lw=1.3, label=t)
s = panel[panel['Ticker'] == BENCH].sort_values('Date')
for w in (50, 200):
    ma = s[PRICE].rolling(w).mean() / s[PRICE].iloc[0]
    ax.plot(s['Date'], ma, ls='--', lw=1.0, label=f'{w}d MA')
ax.set_yscale('log'); ax.legend(fontsize=7, ncol=3)
ax.set_title('Normalised price trends (log scale)')
plt.show()

## Stationarity

**Method** â€” Augmented Dickey-Fuller on price vs. returns

**Finding** â€” Price level p=0.9969 (unit root not rejected, non-stationary); log returns p=4.126e-30 (strongly stationary).

**Modelling decision** â€” Empirical confirmation that returns are the correct modelling target. This is the justification recorded in config.yaml under `target.kind`.


In [ ]:
b = panel[panel['Ticker'] == BENCH].sort_values('Date')
print('ADF price level :', mu.adf_test(b[PRICE]))
print('ADF log returns :', mu.adf_test(mu.log_returns(b[PRICE])))

## Correlation

**Method** â€” Pearson & Spearman across tickers

**Finding** â€” Strong systemic clustering: these large caps do not diversify one another much. The strongest off-diagonal |r| is 0.62.

**Modelling decision** â€” Justifies covariance shrinkage (Ledoit-Wolf) and a per-asset concentration cap in Section 11: a raw sample covariance over near-collinear assets yields unstable, concentrated optimiser weights.


In [ ]:
wide = panel.pivot(index='Date', columns='Ticker', values=PRICE).sort_index()
rets = np.log(wide).diff().dropna(how='all')
sns_corr = rets[UNIVERSE].corr()
plt.figure(figsize=(10, 8))
plt.imshow(sns_corr, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar()
plt.title('Correlation of daily log returns')
plt.show()

## Volatility

**Method** â€” Rolling std, VIX overlay, volatility clustering

**Finding** â€” Volatility clusters into distinct regimes rather than varying independently, and tracks the VIX with a strong positive relationship.

**Modelling decision** â€” Rolling volatility features (Vol_5d .. Vol_63d) and VIX level/change are included in the feature table so models receive regime context, and the risk sub-signal in Section 12 is built from trailing volatility and Beta.


In [ ]:
b = panel[panel['Ticker'] == BENCH].sort_values('Date')
lr = mu.log_returns(b[PRICE])
vol = lr.rolling(21).std() * np.sqrt(252)
v = panel[panel['Ticker'] == VIX].sort_values('Date')
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(vol.index, vol.values, lw=1.2, label='realised 21d vol')
ax.plot(v['Date'], v['Close'] / 100, lw=1.0, alpha=0.8, label=VIX + ' (scaled)')
ax.legend(); ax.set_title('Rolling volatility with VIX overlay')
plt.show()

## Seasonality

**Method** â€” Monthly return averages

**Finding** â€” The strongest month averages +0.188% per day and the weakest -0.028%, a spread of 0.216 percentage points per day.

**Modelling decision** â€” Calendar features (day-of-week, month, turn-of-month) are retained for the models to test, but the spread is small relative to daily dispersion, so no calendar effect is assumed to be an exploitable edge.


In [ ]:
uni = np.log(wide[UNIVERSE]).diff()
heat = uni.groupby([uni.index.year, uni.index.month]).mean().mean(axis=1).unstack()
plt.figure(figsize=(13, 5))
plt.imshow(heat.values * 100, cmap='RdBu_r', aspect='auto')
plt.colorbar(); plt.title('Mean daily return by month x year (%)')
plt.show()

## Distribution

**Method** â€” Return histogram, QQ-plot, skew & kurtosis

**Finding** â€” Negative skew (-0.64) and fat tails (excess kurtosis 15.86); Jarque-Bera p=0.00e+00 rejects normality decisively.

**Modelling decision** â€” Risk metrics must not assume normality. VaR is reported both empirically and under a Student's t fit, maximum drawdown is reported alongside standard deviation, and the deep-learning models use Huber loss to stop return spikes dominating training.


In [ ]:
r = lr.dropna()
print('skew', stats.skew(r), 'excess kurtosis', stats.kurtosis(r))
print('Jarque-Bera', mu.jarque_bera(r))
plt.figure(figsize=(8, 4))
plt.hist(r * 100, bins=120, density=True, alpha=0.6)
xs = np.linspace(r.min(), r.max(), 300) * 100
plt.plot(xs, stats.norm.pdf(xs/100, r.mean(), r.std(ddof=1)), lw=1.8)
plt.title('Return distribution vs normal')
plt.show()

---

*Educational research and decision support only. Not financial advice.*